# Transfer Learning and Fine-Tuning Framework

This notebook implements comprehensive transfer learning strategies including:
- Pre-trained model adaptation
- Progressive unfreezing
- Discriminative learning rates
- Domain adaptation techniques
- Few-shot learning
- Knowledge distillation

In [ ]:
# Import required libraries
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
import torchvision
from torchvision import transforms, models
import timm  # PyTorch Image Models

# Training utilities
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import accuracy_score, f1_score

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Utilities
from typing import Dict, List, Optional, Tuple, Any, Union
from dataclasses import dataclass, field
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Transfer Learning Configuration

In [ ]:
@dataclass
class TransferConfig:
    """Configuration for transfer learning."""
    # Model settings
    model_name: str = 'resnet50'
    pretrained: bool = True
    freeze_backbone: bool = True
    
    # Fine-tuning strategy
    fine_tune_strategy: str = 'progressive'  # 'all', 'last_layer', 'progressive'
    unfreeze_schedule: List[int] = field(default_factory=lambda: [5, 10, 15])
    discriminative_lr: bool = True
    lr_mult: float = 0.1  # LR multiplier for earlier layers
    
    # Training settings
    batch_size: int = 32
    base_lr: float = 1e-3
    weight_decay: float = 1e-4
    max_epochs: int = 100
    patience: int = 10
    
    # Data augmentation
    use_augmentation: bool = True
    mixup_alpha: float = 0.2
    cutmix_alpha: float = 1.0
    
    # Domain adaptation
    use_domain_adaptation: bool = False
    adaptation_method: str = 'dann'  # 'dann', 'coral', 'mmd'
    

@dataclass
class ModelRegistry:
    """Registry of available pre-trained models."""
    
    vision_models = {
        # Torchvision models
        'resnet18': models.resnet18,
        'resnet34': models.resnet34,
        'resnet50': models.resnet50,
        'resnet101': models.resnet101,
        'densenet121': models.densenet121,
        'densenet169': models.densenet169,
        'vgg16': models.vgg16,
        'vgg19': models.vgg19,
        'mobilenet_v2': models.mobilenet_v2,
        'efficientnet_b0': models.efficientnet_b0,
        'efficientnet_b7': models.efficientnet_b7,
        
        # TIMM models (extended)
        'vit_base': 'vit_base_patch16_224',
        'vit_large': 'vit_large_patch16_224',
        'swin_base': 'swin_base_patch4_window7_224',
        'convnext_base': 'convnext_base',
    }
    
    text_models = {
        'bert': 'bert-base-uncased',
        'roberta': 'roberta-base',
        'distilbert': 'distilbert-base-uncased',
    }

## 2. Transfer Learning Models

In [ ]:
class TransferLearningModel(nn.Module):
    """Base class for transfer learning models."""
    
    def __init__(self, 
                 backbone: nn.Module,
                 num_classes: int,
                 feature_dim: Optional[int] = None,
                 dropout: float = 0.2):
        """
        Initialize transfer learning model.
        
        Parameters:
        -----------
        backbone : nn.Module
            Pre-trained backbone model
        num_classes : int
            Number of output classes
        feature_dim : int
            Dimension of features from backbone
        dropout : float
            Dropout probability
        """
        super().__init__()
        self.backbone = backbone
        self.num_classes = num_classes
        
        # Determine feature dimension
        if feature_dim is None:
            feature_dim = self._get_feature_dim()
        
        # Create new classifier
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
        
        # Initialize classifier weights
        self._init_classifier()
    
    def _get_feature_dim(self) -> int:
        """Get feature dimension from backbone."""
        # Try to infer from common architectures
        if hasattr(self.backbone, 'fc'):
            return self.backbone.fc.in_features
        elif hasattr(self.backbone, 'classifier'):
            if isinstance(self.backbone.classifier, nn.Sequential):
                return self.backbone.classifier[-1].in_features
            else:
                return self.backbone.classifier.in_features
        elif hasattr(self.backbone, 'head'):
            return self.backbone.head.in_features
        else:
            # Default
            return 2048
    
    def _init_classifier(self):
        """Initialize classifier weights."""
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass."""
        features = self.extract_features(x)
        return self.classifier(features)
    
    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        """Extract features from backbone."""
        # Remove original classifier from backbone
        if hasattr(self.backbone, 'fc'):
            self.backbone.fc = nn.Identity()
        elif hasattr(self.backbone, 'classifier'):
            self.backbone.classifier = nn.Identity()
        elif hasattr(self.backbone, 'head'):
            self.backbone.head = nn.Identity()
        
        return self.backbone(x)
    
    def freeze_backbone(self):
        """Freeze backbone parameters."""
        for param in self.backbone.parameters():
            param.requires_grad = False
    
    def unfreeze_backbone(self):
        """Unfreeze backbone parameters."""
        for param in self.backbone.parameters():
            param.requires_grad = True
    
    def progressive_unfreeze(self, layer_group: int):
        """Progressive unfreezing of layers."""
        # Get layer groups
        layer_groups = self._get_layer_groups()
        
        # Unfreeze specified group and later
        for i in range(layer_group, len(layer_groups)):
            for param in layer_groups[i].parameters():
                param.requires_grad = True
    
    def _get_layer_groups(self) -> List[nn.Module]:
        """Get layer groups for progressive unfreezing."""
        # This is model-specific; override in subclasses
        if hasattr(self.backbone, 'layer4'):
            # ResNet-like
            return [self.backbone.layer1, self.backbone.layer2,
                   self.backbone.layer3, self.backbone.layer4]
        else:
            # Generic: split into 4 groups
            modules = list(self.backbone.children())
            n = len(modules)
            return [nn.Sequential(*modules[i:i+n//4]) 
                   for i in range(0, n, n//4)]


class VisionTransferModel(TransferLearningModel):
    """Transfer learning model for vision tasks."""
    
    @classmethod
    def from_pretrained(cls, model_name: str, num_classes: int, 
                       pretrained: bool = True, **kwargs):
        """Create model from pre-trained backbone."""
        # Get model from registry
        if model_name in ModelRegistry.vision_models:
            model_fn = ModelRegistry.vision_models[model_name]
            
            if isinstance(model_fn, str):
                # TIMM model
                backbone = timm.create_model(model_fn, pretrained=pretrained)
            else:
                # Torchvision model
                backbone = model_fn(pretrained=pretrained)
        else:
            raise ValueError(f"Unknown model: {model_name}")
        
        return cls(backbone, num_classes, **kwargs)

## 3. Fine-Tuning Strategies

In [ ]:
class FineTuningStrategy:
    """Base class for fine-tuning strategies."""
    
    def __init__(self, model: TransferLearningModel, config: TransferConfig):
        self.model = model
        self.config = config
    
    def setup_optimizer(self) -> torch.optim.Optimizer:
        """Setup optimizer with appropriate parameter groups."""
        raise NotImplementedError
    
    def on_epoch_end(self, epoch: int):
        """Called at the end of each epoch."""
        pass


class ProgressiveUnfreezing(FineTuningStrategy):
    """Progressive unfreezing strategy."""
    
    def setup_optimizer(self) -> torch.optim.Optimizer:
        """Setup optimizer with discriminative learning rates."""
        # Initially freeze backbone
        self.model.freeze_backbone()
        
        # Get parameter groups
        param_groups = self._get_param_groups()
        
        # Create optimizer
        optimizer = optim.AdamW(param_groups, weight_decay=self.config.weight_decay)
        
        return optimizer
    
    def _get_param_groups(self) -> List[Dict]:
        """Get parameter groups with discriminative learning rates."""
        param_groups = []
        
        if self.config.discriminative_lr:
            # Backbone layers with lower learning rate
            layer_groups = self.model._get_layer_groups()
            
            for i, layer_group in enumerate(layer_groups):
                lr_scale = self.config.lr_mult ** (len(layer_groups) - i - 1)
                param_groups.append({
                    'params': layer_group.parameters(),
                    'lr': self.config.base_lr * lr_scale
                })
        else:
            # Backbone with single learning rate
            param_groups.append({
                'params': self.model.backbone.parameters(),
                'lr': self.config.base_lr * self.config.lr_mult
            })
        
        # Classifier with base learning rate
        param_groups.append({
            'params': self.model.classifier.parameters(),
            'lr': self.config.base_lr
        })
        
        return param_groups
    
    def on_epoch_end(self, epoch: int):
        """Progressive unfreezing based on schedule."""
        if epoch in self.config.unfreeze_schedule:
            layer_idx = self.config.unfreeze_schedule.index(epoch)
            print(f"Unfreezing layer group {layer_idx}")
            self.model.progressive_unfreeze(layer_idx)


class GradualUnfreezing(FineTuningStrategy):
    """Gradual unfreezing with warm-up."""
    
    def __init__(self, model: TransferLearningModel, config: TransferConfig, 
                 warmup_epochs: int = 5):
        super().__init__(model, config)
        self.warmup_epochs = warmup_epochs
        self.current_epoch = 0
    
    def setup_optimizer(self) -> torch.optim.Optimizer:
        """Setup optimizer for gradual unfreezing."""
        # Start with frozen backbone
        self.model.freeze_backbone()
        
        # All parameters in optimizer but with different states
        optimizer = optim.AdamW(
            self.model.parameters(),
            lr=self.config.base_lr,
            weight_decay=self.config.weight_decay
        )
        
        return optimizer
    
    def on_epoch_end(self, epoch: int):
        """Gradually unfreeze layers."""
        self.current_epoch = epoch
        
        if epoch == self.warmup_epochs:
            print("Unfreezing all layers after warm-up")
            self.model.unfreeze_backbone()
        elif epoch > self.warmup_epochs:
            # Optionally adjust learning rates
            pass

## 4. Domain Adaptation

In [ ]:
class DomainAdaptation(nn.Module):
    """Domain adaptation techniques."""
    
    def __init__(self, feature_extractor: nn.Module, 
                 feature_dim: int,
                 num_classes: int,
                 adaptation_method: str = 'dann'):
        """
        Initialize domain adaptation model.
        
        Parameters:
        -----------
        feature_extractor : nn.Module
            Feature extraction network
        feature_dim : int
            Dimension of features
        num_classes : int
            Number of classes
        adaptation_method : str
            Adaptation method ('dann', 'coral', 'mmd')
        """
        super().__init__()
        self.feature_extractor = feature_extractor
        self.adaptation_method = adaptation_method
        
        # Task classifier
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        
        if adaptation_method == 'dann':
            # Domain discriminator for DANN
            self.domain_discriminator = nn.Sequential(
                nn.Linear(feature_dim, 256),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(128, 1)
            )
    
    def forward(self, x_source: torch.Tensor, 
               x_target: Optional[torch.Tensor] = None,
               alpha: float = 1.0) -> Dict[str, torch.Tensor]:
        """Forward pass with domain adaptation."""
        # Extract features
        features_source = self.feature_extractor(x_source)
        
        # Task predictions
        task_output = self.classifier(features_source)
        
        outputs = {'task_output': task_output}
        
        if x_target is not None and self.adaptation_method == 'dann':
            # DANN: Domain adversarial training
            features_target = self.feature_extractor(x_target)
            
            # Gradient reversal
            features_source_rev = GradientReversal.apply(features_source, alpha)
            features_target_rev = GradientReversal.apply(features_target, alpha)
            
            # Domain predictions
            domain_output_source = self.domain_discriminator(features_source_rev)
            domain_output_target = self.domain_discriminator(features_target_rev)
            
            outputs['domain_output_source'] = domain_output_source
            outputs['domain_output_target'] = domain_output_target
            
        elif x_target is not None and self.adaptation_method == 'coral':
            # CORAL: Correlation alignment
            features_target = self.feature_extractor(x_target)
            coral_loss = self.coral_loss(features_source, features_target)
            outputs['coral_loss'] = coral_loss
            
        elif x_target is not None and self.adaptation_method == 'mmd':
            # MMD: Maximum mean discrepancy
            features_target = self.feature_extractor(x_target)
            mmd_loss = self.mmd_loss(features_source, features_target)
            outputs['mmd_loss'] = mmd_loss
        
        return outputs
    
    def coral_loss(self, source: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """CORAL loss for domain adaptation."""
        d = source.size(1)
        
        # Covariance matrices
        source_cov = self._compute_covariance(source)
        target_cov = self._compute_covariance(target)
        
        # Frobenius norm
        loss = torch.norm(source_cov - target_cov, p='fro') / (4 * d * d)
        
        return loss
    
    def mmd_loss(self, source: torch.Tensor, target: torch.Tensor, 
                 kernel: str = 'rbf') -> torch.Tensor:
        """Maximum Mean Discrepancy loss."""
        if kernel == 'rbf':
            # RBF kernel
            bandwidth = self._median_heuristic(source, target)
            
            XX = self._rbf_kernel(source, source, bandwidth)
            YY = self._rbf_kernel(target, target, bandwidth)
            XY = self._rbf_kernel(source, target, bandwidth)
            
            mmd = XX.mean() + YY.mean() - 2 * XY.mean()
        else:
            # Linear kernel
            mmd = (source.mean(0) - target.mean(0)).pow(2).sum()
        
        return mmd
    
    def _compute_covariance(self, x: torch.Tensor) -> torch.Tensor:
        """Compute covariance matrix."""
        n = x.size(0)
        x_mean = x.mean(0, keepdim=True)
        x_centered = x - x_mean
        cov = x_centered.t() @ x_centered / (n - 1)
        return cov
    
    def _rbf_kernel(self, x: torch.Tensor, y: torch.Tensor, 
                   bandwidth: float) -> torch.Tensor:
        """RBF kernel computation."""
        pairwise_distances = torch.cdist(x, y, p=2)
        kernel = torch.exp(-pairwise_distances.pow(2) / (2 * bandwidth ** 2))
        return kernel
    
    def _median_heuristic(self, x: torch.Tensor, y: torch.Tensor) -> float:
        """Median heuristic for bandwidth selection."""
        with torch.no_grad():
            combined = torch.cat([x, y], dim=0)
            distances = torch.cdist(combined, combined, p=2)
            median_dist = distances.median().item()
        return median_dist / np.sqrt(2)


class GradientReversal(torch.autograd.Function):
    """Gradient reversal layer for DANN."""
    
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

## 5. Knowledge Distillation

In [ ]:
class KnowledgeDistillation:
    """Knowledge distillation for model compression."""
    
    def __init__(self, teacher_model: nn.Module, 
                 student_model: nn.Module,
                 temperature: float = 3.0,
                 alpha: float = 0.7):
        """
        Initialize knowledge distillation.
        
        Parameters:
        -----------
        teacher_model : nn.Module
            Pre-trained teacher model
        student_model : nn.Module
            Student model to be trained
        temperature : float
            Temperature for softening probabilities
        alpha : float
            Weight for distillation loss
        """
        self.teacher_model = teacher_model
        self.student_model = student_model
        self.temperature = temperature
        self.alpha = alpha
        
        # Freeze teacher model
        self.teacher_model.eval()
        for param in self.teacher_model.parameters():
            param.requires_grad = False
    
    def distillation_loss(self, student_logits: torch.Tensor,
                         teacher_logits: torch.Tensor,
                         labels: torch.Tensor) -> torch.Tensor:
        """Calculate distillation loss."""
        # Soft targets from teacher
        teacher_probs = F.softmax(teacher_logits / self.temperature, dim=1)
        
        # Distillation loss
        distill_loss = F.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=1),
            teacher_probs,
            reduction='batchmean'
        ) * (self.temperature ** 2)
        
        # Student loss
        student_loss = F.cross_entropy(student_logits, labels)
        
        # Combined loss
        total_loss = self.alpha * distill_loss + (1 - self.alpha) * student_loss
        
        return total_loss, distill_loss, student_loss
    
    def train_step(self, x: torch.Tensor, y: torch.Tensor,
                  optimizer: torch.optim.Optimizer) -> Dict[str, float]:
        """Single training step with distillation."""
        # Teacher predictions
        with torch.no_grad():
            teacher_logits = self.teacher_model(x)
        
        # Student predictions
        student_logits = self.student_model(x)
        
        # Calculate loss
        total_loss, distill_loss, student_loss = self.distillation_loss(
            student_logits, teacher_logits, y
        )
        
        # Backward pass
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        return {
            'total_loss': total_loss.item(),
            'distill_loss': distill_loss.item(),
            'student_loss': student_loss.item()
        }

## 6. Few-Shot Learning

In [ ]:
class PrototypicalNetwork(nn.Module):
    """Prototypical Networks for few-shot learning."""
    
    def __init__(self, encoder: nn.Module, feature_dim: int):
        """
        Initialize Prototypical Network.
        
        Parameters:
        -----------
        encoder : nn.Module
            Feature encoder network
        feature_dim : int
            Dimension of encoded features
        """
        super().__init__()
        self.encoder = encoder
        self.feature_dim = feature_dim
    
    def compute_prototypes(self, support_features: torch.Tensor,
                          support_labels: torch.Tensor) -> torch.Tensor:
        """Compute class prototypes from support set."""
        n_classes = support_labels.unique().size(0)
        prototypes = torch.zeros(n_classes, self.feature_dim).to(support_features.device)
        
        for c in range(n_classes):
            mask = support_labels == c
            prototypes[c] = support_features[mask].mean(dim=0)
        
        return prototypes
    
    def forward(self, support_set: Tuple[torch.Tensor, torch.Tensor],
               query_set: torch.Tensor) -> torch.Tensor:
        """Forward pass for few-shot classification."""
        support_data, support_labels = support_set
        
        # Encode support and query sets
        support_features = self.encoder(support_data)
        query_features = self.encoder(query_set)
        
        # Compute prototypes
        prototypes = self.compute_prototypes(support_features, support_labels)
        
        # Calculate distances to prototypes
        distances = torch.cdist(query_features, prototypes, p=2)
        
        # Convert to similarities (negative distances)
        logits = -distances
        
        return logits


class MAML(nn.Module):
    """Model-Agnostic Meta-Learning."""
    
    def __init__(self, model: nn.Module, 
                 inner_lr: float = 0.01,
                 inner_steps: int = 5):
        """
        Initialize MAML.
        
        Parameters:
        -----------
        model : nn.Module
            Base model
        inner_lr : float
            Learning rate for inner loop
        inner_steps : int
            Number of inner loop steps
        """
        super().__init__()
        self.model = model
        self.inner_lr = inner_lr
        self.inner_steps = inner_steps
    
    def inner_loop(self, support_data: torch.Tensor,
                  support_labels: torch.Tensor,
                  params: Optional[Dict] = None) -> Dict:
        """Inner loop adaptation."""
        if params is None:
            params = {name: param.clone() for name, param in self.model.named_parameters()}
        
        for _ in range(self.inner_steps):
            # Forward pass with current params
            logits = self.functional_forward(support_data, params)
            loss = F.cross_entropy(logits, support_labels)
            
            # Compute gradients
            grads = torch.autograd.grad(loss, params.values(), create_graph=True)
            
            # Update parameters
            params = {name: param - self.inner_lr * grad
                     for (name, param), grad in zip(params.items(), grads)}
        
        return params
    
    def functional_forward(self, x: torch.Tensor, params: Dict) -> torch.Tensor:
        """Forward pass with given parameters."""
        # This is a simplified version; real implementation would need
        # to handle all layer types properly
        for name, param in params.items():
            # Apply parameters to model
            pass
        return self.model(x)
    
    def forward(self, support_set: Tuple[torch.Tensor, torch.Tensor],
               query_set: Tuple[torch.Tensor, torch.Tensor]) -> torch.Tensor:
        """Meta-learning forward pass."""
        support_data, support_labels = support_set
        query_data, query_labels = query_set
        
        # Inner loop adaptation
        adapted_params = self.inner_loop(support_data, support_labels)
        
        # Evaluation on query set
        query_logits = self.functional_forward(query_data, adapted_params)
        query_loss = F.cross_entropy(query_logits, query_labels)
        
        return query_loss

## 7. Transfer Learning Pipeline

In [ ]:
class TransferLearningPipeline:
    """Complete transfer learning pipeline."""
    
    def __init__(self, config: TransferConfig):
        self.config = config
        self.model = None
        self.optimizer = None
        self.scheduler = None
        self.strategy = None
        self.history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    
    def setup_model(self, num_classes: int):
        """Setup transfer learning model."""
        self.model = VisionTransferModel.from_pretrained(
            self.config.model_name,
            num_classes=num_classes,
            pretrained=self.config.pretrained
        )
        
        if self.config.freeze_backbone:
            self.model.freeze_backbone()
    
    def setup_training(self):
        """Setup training components."""
        # Setup strategy
        if self.config.fine_tune_strategy == 'progressive':
            self.strategy = ProgressiveUnfreezing(self.model, self.config)
        else:
            self.strategy = GradualUnfreezing(self.model, self.config)
        
        # Setup optimizer
        self.optimizer = self.strategy.setup_optimizer()
        
        # Setup scheduler
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer,
            T_max=self.config.max_epochs
        )
    
    def train_epoch(self, train_loader: DataLoader, 
                   epoch: int) -> float:
        """Train for one epoch."""
        self.model.train()
        total_loss = 0
        
        for batch_idx, (data, target) in enumerate(tqdm(train_loader)):
            data = data.cuda() if torch.cuda.is_available() else data
            target = target.cuda() if torch.cuda.is_available() else target
            
            self.optimizer.zero_grad()
            output = self.model(data)
            loss = F.cross_entropy(output, target)
            
            loss.backward()
            self.optimizer.step()
            
            total_loss += loss.item()
        
        return total_loss / len(train_loader)
    
    def validate(self, val_loader: DataLoader) -> Tuple[float, float]:
        """Validate model."""
        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for data, target in val_loader:
                data = data.cuda() if torch.cuda.is_available() else data
                target = target.cuda() if torch.cuda.is_available() else target
                
                output = self.model(data)
                loss = F.cross_entropy(output, target)
                
                total_loss += loss.item()
                _, predicted = output.max(1)
                total += target.size(0)
                correct += predicted.eq(target).sum().item()
        
        return total_loss / len(val_loader), correct / total
    
    def train(self, train_loader: DataLoader, val_loader: DataLoader):
        """Complete training loop."""
        best_acc = 0
        patience_counter = 0
        
        for epoch in range(self.config.max_epochs):
            print(f"\nEpoch {epoch+1}/{self.config.max_epochs}")
            
            # Training
            train_loss = self.train_epoch(train_loader, epoch)
            
            # Validation
            val_loss, val_acc = self.validate(val_loader)
            
            # Update history
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)
            
            print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
            
            # Learning rate scheduling
            self.scheduler.step()
            
            # Strategy updates (e.g., unfreezing)
            self.strategy.on_epoch_end(epoch)
            
            # Early stopping
            if val_acc > best_acc:
                best_acc = val_acc
                patience_counter = 0
                # Save best model
                torch.save(self.model.state_dict(), 'best_model.pt')
            else:
                patience_counter += 1
                if patience_counter >= self.config.patience:
                    print("Early stopping triggered")
                    break
        
        print(f"\nBest validation accuracy: {best_acc:.4f}")

## 8. Example Usage

In [ ]:
# Example: Transfer learning with progressive unfreezing
print("Setting up transfer learning example...\n")

# Configuration
config = TransferConfig(
    model_name='resnet50',
    pretrained=True,
    freeze_backbone=True,
    fine_tune_strategy='progressive',
    unfreeze_schedule=[5, 10, 15],
    discriminative_lr=True,
    lr_mult=0.1,
    batch_size=32,
    base_lr=1e-3,
    max_epochs=20
)

# Create synthetic dataset
from torchvision.datasets import FakeData
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Create fake datasets for demonstration
train_dataset = FakeData(size=500, image_size=(3, 224, 224),
                        num_classes=10, transform=transform)
val_dataset = FakeData(size=100, image_size=(3, 224, 224),
                      num_classes=10, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

print(f"Dataset created:")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")
print(f"  Number of classes: 10")

In [ ]:
# Initialize pipeline
pipeline = TransferLearningPipeline(config)

# Setup model
pipeline.setup_model(num_classes=10)
print(f"\nModel: {config.model_name}")
print(f"Pretrained: {config.pretrained}")
print(f"Frozen backbone: {config.freeze_backbone}")

# Setup training
pipeline.setup_training()
print(f"\nTraining setup complete")
print(f"Strategy: {config.fine_tune_strategy}")
print(f"Discriminative LR: {config.discriminative_lr}")

In [ ]:
# Train model (reduced epochs for demonstration)
print("\nStarting training...")
config.max_epochs = 3  # Reduce for demo
pipeline.config = config

# Run training
pipeline.train(train_loader, val_loader)

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
ax1.plot(pipeline.history['train_loss'], label='Train Loss')
ax1.plot(pipeline.history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy plot
ax2.plot(pipeline.history['val_acc'], label='Val Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Summary

This notebook provides a comprehensive transfer learning framework with:

### Key Features:
1. **Pre-trained Model Adaptation**: Support for multiple architectures
2. **Fine-tuning Strategies**:
   - Progressive unfreezing
   - Gradual unfreezing with warm-up
   - Discriminative learning rates

3. **Domain Adaptation**:
   - DANN (Domain Adversarial Neural Networks)
   - CORAL (Correlation Alignment)
   - MMD (Maximum Mean Discrepancy)

4. **Knowledge Distillation**: Teacher-student learning

5. **Few-Shot Learning**:
   - Prototypical Networks
   - MAML (Model-Agnostic Meta-Learning)

### Applications:
- Computer vision tasks with limited data
- Domain adaptation across datasets
- Model compression via distillation
- Few-shot classification
- Fine-tuning for specific tasks

The framework is modular and can be easily extended with new models, strategies, and techniques.